# Pipeline Medallion com PySpark
## Análise de Incidentes de Cibersegurança — Bronze → Prata → Ouro → ML
### Aluno: Benjamin Yuji Suzuki


In [1]:
import time
import os
import json
import warnings
import numpy as np
import pandas as pd
import pyarrow  # for parquet
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

# ── Inicializar Spark ──
spark = SparkSession.builder \
    .appName("PipelineMedallion") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark OK:", spark.version)

# Diretorio base do projeto PySpark
BASE = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
CAMADA_INICIAL = os.path.join(BASE, "..", "projeto_rust", "camada_inicial")
COMPARACAO = os.path.join(BASE, "..", "comparacao_tempos")



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/24 07:29:53 WARN Utils: Your hostname, ben, resolves to a loopback address: 127.0.1.1; using 192.168.5.123 instead (on interface wlp63s0)
26/05/24 07:29:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/24 07:29:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark OK: 4.1.2


In [2]:
def setup_dirs():
    for d in ["camada_bronze", "camada_prata", "camada_ouro", "graficos", "previsoes", "modelos"]:
        os.makedirs(d, exist_ok=True)
        print(f"  Diretorio garantido: {d}/")
    os.makedirs(COMPARACAO, exist_ok=True)

setup_dirs()


  Diretorio garantido: camada_bronze/
  Diretorio garantido: camada_prata/
  Diretorio garantido: camada_ouro/
  Diretorio garantido: graficos/
  Diretorio garantido: previsoes/
  Diretorio garantido: modelos/


## 🥉 Etapa 1: Camada Bronze — Ingestão de Dados Brutos
**Objetivo:** Preservar os dados em estado bruto (imutável) em formato colunar (Parquet).
Adicionamos 4 colunas de auditoria para rastreabilidade: meta_arquivo_origem, meta_qtd_linhas, meta_hash_ingestao, meta_data_carga.


**Arquivos de entrada:**
- `financial_impact.csv` → 778 registros de perdas financeiras
- `incidents_master.csv` → 850 registros de ataques cibernéticos
- `market_impact.csv` → 358 registros de impacto no mercado


In [3]:
tempos = {}  # dicionario para armazenar tempos de cada etapa

# ── BRONZE ──
print("=" * 60)
print("  BRONZE: INGESTAO DE DADOS BRUTOS")
print("=" * 60)
t0 = time.time()

arquivos = [
    ("financial_impact.csv", os.path.join(CAMADA_INICIAL, "financial_impact.csv")),
    ("incidents_master.csv", os.path.join(CAMADA_INICIAL, "incidents_master.csv")),
    ("market_impact.csv",    os.path.join(CAMADA_INICIAL, "market_impact.csv")),
]

for nome, caminho in arquivos:
    print(f"\n  Lendo: {caminho}")
    df = spark.read.option("header", "true").csv(caminho)
    linhas = df.count()
    nome_parquet = f"camada_bronze/{nome.replace('.csv', '')}.parquet"

    df_bronze = df.withColumn("meta_arquivo_origem", F.lit(nome)) \
                  .withColumn("meta_qtd_linhas", F.lit(linhas)) \
                  .withColumn("meta_hash_ingestao", F.expr("uuid()")) \
                  .withColumn("meta_data_carga", F.current_timestamp())

    df_bronze.write.mode("overwrite").parquet(nome_parquet)
    print(f"  {nome} -> {nome_parquet} ({linhas} linhas)")

# Relatorio
print("\n--- RELATORIO DA CAMADA BRONZE ---")
for nome, _ in arquivos:
    nome_parquet = f"camada_bronze/{nome.replace('.csv', '')}.parquet"
    df = spark.read.parquet(nome_parquet)
    print(f"  {nome.replace('.csv', ''):<20} | Linhas: {df.count():>5} | Colunas: {len(df.columns):>3}")

bronze_time = time.time() - t0
tempos["Bronze"] = bronze_time
print(f"\n  [BRONZE] Concluida em {bronze_time:.4f}s\n")


  BRONZE: INGESTAO DE DADOS BRUTOS

  Lendo: /home/ben/Área de trabalho/Projeto Ciencia de dados/projeto_pyspark/../projeto_rust/camada_inicial/financial_impact.csv


  financial_impact.csv -> camada_bronze/financial_impact.parquet (778 linhas)

  Lendo: /home/ben/Área de trabalho/Projeto Ciencia de dados/projeto_pyspark/../projeto_rust/camada_inicial/incidents_master.csv


26/05/24 07:30:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  incidents_master.csv -> camada_bronze/incidents_master.parquet (850 linhas)

  Lendo: /home/ben/Área de trabalho/Projeto Ciencia de dados/projeto_pyspark/../projeto_rust/camada_inicial/market_impact.csv


  market_impact.csv -> camada_bronze/market_impact.parquet (358 linhas)

--- RELATORIO DA CAMADA BRONZE ---


  financial_impact     | Linhas:   778 | Colunas:  23


  incidents_master     | Linhas:   850 | Colunas:  36


  market_impact        | Linhas:   358 | Colunas:  35

  [BRONZE] Concluida em 9.3323s



## 🥈 Etapa 2: Camada Prata (Silver) — Limpeza e Integração
**Objetivo:** Limpar, integrar e preparar os dados. Usamos INNER JOIN entre incidents_master e financial_impact.
Aplicamos anti-data leakage removendo colunas que vazariam informação do futuro.


**Data Leakage — Conceito Crítico:**
Data Leakage ocorre quando usamos informação que NÃO estaria disponível no momento da predição.
Colunas removidas: `disclosure_date` (só conhecida após o ataque), `downtime_hours` (medido depois),
`company_name`/`stock_ticker` (identificadores únicos), `notes`/`created_at`/`updated_at` (metadados),
`industry_secondary`/`attack_vector_secondary` (redundantes), `meta_*` (auditoria interna).


In [4]:
# ── PRATA ──
print("=" * 60)
print("  PRATA: LIMPEZA E INTEGRACAO")
print("=" * 60)
t0 = time.time()

df_incidentes = spark.read.parquet("camada_bronze/incidents_master.parquet")
df_financeiro = spark.read.parquet("camada_bronze/financial_impact.parquet") \
                        .select("incident_id", "total_loss_usd")

# INNER JOIN: apenas incidentes com registro financeiro
df_prata = df_incidentes.join(df_financeiro, on="incident_id", how="inner")

colunas_para_dropar = [
    "downtime_hours", "data_compromised_records", "disclosure_date",
    "company_name", "stock_ticker", "notes", "created_at", "updated_at",
    "industry_secondary", "attack_vector_secondary", "review_flag",
    "meta_arquivo_origem", "meta_qtd_linhas", "meta_hash_ingestao", "meta_data_carga",
    "incident_id",
]
colunas_existentes = [c for c in colunas_para_dropar if c in df_prata.columns]
df_prata = df_prata.drop(*colunas_existentes)

# Filtrar nulos no target
df_prata = df_prata.filter(F.col("total_loss_usd").isNotNull())

# Tratar nulos em colunas categoricas
colunas_categoricas = [
    "attack_chain", "attributed_group", "attribution_confidence",
    "data_type", "data_source_secondary", "attack_vector_primary", "data_source_primary",
]
for c in colunas_categoricas:
    if c in df_prata.columns:
        df_prata = df_prata.fillna({c: "Desconhecido"})

from pyspark.sql.functions import col
df_prata = df_prata.withColumn("total_loss_usd", col("total_loss_usd").cast("double"))
df_prata = df_prata.withColumn("company_revenue_usd", col("company_revenue_usd").cast("double"))
df_prata = df_prata.withColumn("employee_count", col("employee_count").cast("double"))
df_prata = df_prata.withColumn("quality_score", col("quality_score").cast("double"))
df_prata = df_prata.withColumn("confidence_tier", col("confidence_tier").cast("double"))
df_prata.write.mode("overwrite").parquet("camada_prata/dataset_ml.parquet")

n_linhas = df_prata.count()
print(f"\n  Prata salvo: camada_prata/dataset_ml.parquet ({n_linhas} linhas x {len(df_prata.columns)} colunas)")

# Relatorio de qualidade
print("\n--- RELATORIO DE QUALIDADE — CAMADA PRATA ---")
for c in df_prata.columns:
    nulos = df_prata.filter(F.col(c).isNull()).count()
    if nulos > 0:
        print(f"  {c}: {nulos} nulos ({100.0 * nulos / n_linhas:.1f}%)")
print("  Dataset sem nulos (todos tratados)!")

prata_time = time.time() - t0
tempos["Prata"] = prata_time
print(f"\n  [PRATA] Concluida em {prata_time:.4f}s\n")


  PRATA: LIMPEZA E INTEGRACAO



  Prata salvo: camada_prata/dataset_ml.parquet (778 linhas x 21 colunas)

--- RELATORIO DE QUALIDADE — CAMADA PRATA ---


  Dataset sem nulos (todos tratados)!

  [PRATA] Concluida em 7.1001s



## 📊 Etapa 3: EDA — Análise Exploratória de Dados
**Hipóteses testadas:**
1. **H1:** Indústrias de tecnologia (código 51) sofrem mais ataques que outros setores
2. **H2:** Ransomware causa o maior prejuízo financeiro médio
3. **H3:** Dados mistos (mixed) são os mais visados por atacantes

**Gráficos adicionais:**
- G4: Distribuição do prejuízo total (histograma)
- G5: Análise de outliers (IQR scatter)
- G6: Matriz de correlação entre variáveis numéricas


In [5]:
# ── EDA ──
print("=" * 60)
print("  EDA: ANALISE EXPLORATORIA DE DADOS")
print("=" * 60)
t0 = time.time()

from pyspark.sql.functions import col
df_prata = spark.read.parquet("camada_prata/dataset_ml.parquet")
df_prata = df_prata.withColumn("total_loss_usd", col("total_loss_usd").cast("double"))
df_prata = df_prata.withColumn("company_revenue_usd", col("company_revenue_usd").cast("double"))
df_prata = df_prata.withColumn("employee_count", col("employee_count").cast("double"))
df_prata = df_prata.withColumn("quality_score", col("quality_score").cast("double"))
df_prata = df_prata.withColumn("confidence_tier", col("confidence_tier").cast("double"))
df_pd = df_prata.toPandas()

print(f"  Dataset: {len(df_pd)} linhas x {len(df_pd.columns)} colunas\n")

# ---- H1: Top 10 Industrias com Maior Volume de Ataques ----
print("H1: Industrias de tecnologia sofrem mais ataques que outros setores?")
h1 = df_pd.groupby("industry_primary").size().reset_index(name="count") \
          .sort_values("count", ascending=False).head(10)

fig, ax = plt.subplots()
bars = ax.barh(range(len(h1)), h1["count"].values, color=plt.cm.Blues(np.linspace(0.4, 0.9, len(h1))))
ax.set_yticks(range(len(h1)))
ax.set_yticklabels(h1["industry_primary"].values)
ax.set_xlabel("Frequencia")
ax.set_title("Top 10 Industrias com Maior Volume de Ataques", fontweight="bold", fontsize=14)
ax.text(0.5, -0.12, "H1: Setores com mais incidentes de ciberseguranca registrados",
        transform=ax.transAxes, ha="center", fontsize=10, color="gray", style="italic")
sns.despine()
plt.tight_layout()
plt.savefig("graficos/grafico1_top_industrias.png", bbox_inches="tight")
plt.close()
print("  Grafico salvo: graficos/grafico1_top_industrias.png")
print("  INTERPRETACAO: Se a industria 51 (technology) ou 52 (finance)")
print("  liderar, confirma-se que setores com muitos dados digitais")
print("  sao alvos prioritarios.\n")

# ---- H2: Prejuizo Medio por Vetor de Ataque ----
print("H2: Ransomware causa o maior prejuizo financeiro medio?")
h2 = df_pd.groupby("attack_vector_primary")["total_loss_usd"].mean() \
          .reset_index().sort_values("total_loss_usd", ascending=False).head(10)

fig, ax = plt.subplots()
cores = plt.cm.Reds(np.linspace(0.4, 0.9, len(h2)))
bars = ax.barh(range(len(h2)), h2["total_loss_usd"].values, color=cores)
ax.set_yticks(range(len(h2)))
ax.set_yticklabels(h2["attack_vector_primary"].values)
ax.set_xlabel("Prejuizo Medio (USD)")
ax.set_title("Prejuizo Medio por Vetor de Ataque (USD)", fontweight="bold", fontsize=14)
ax.text(0.5, -0.12, "H2: Qual tipo de ataque causa maior dano financeiro medio?",
        transform=ax.transAxes, ha="center", fontsize=10, color="gray", style="italic")
sns.despine()
plt.tight_layout()
plt.savefig("graficos/grafico2_prejuizo_vetor.png", bbox_inches="tight")
plt.close()
print("  Grafico salvo: graficos/grafico2_prejuizo_vetor.png")
print("  INTERPRETACAO: Se ransomware lidera, justifica-se investir")
print("  em protecao especifica (backups off-site, treinamento anti-phishing).\n")

# ---- H3: Frequencia de Tipos de Dados Roubados ----
print("H3: Dados mistos (mixed) sao os mais visados por atacantes?")
h3 = df_pd.groupby("data_type").size().reset_index(name="count") \
          .sort_values("count", ascending=False).head(10)

fig, ax = plt.subplots()
cores = plt.cm.Greens(np.linspace(0.3, 0.9, len(h3)))
ax.barh(range(len(h3)), h3["count"].values, color=cores)
ax.set_yticks(range(len(h3)))
ax.set_yticklabels(h3["data_type"].values)
ax.set_xlabel("Frequencia")
ax.set_title("Frequencia de Tipos de Dados Roubados", fontweight="bold", fontsize=14)
ax.text(0.5, -0.12, "H3: Que tipo de dado os atacantes mais visam?",
        transform=ax.transAxes, ha="center", fontsize=10, color="gray", style="italic")
sns.despine()
plt.tight_layout()
plt.savefig("graficos/grafico3_tipos_dados.png", bbox_inches="tight")
plt.close()
print("  Grafico salvo: graficos/grafico3_tipos_dados.png")
print("  INTERPRETACAO: Dados financeiros + pessoais (mixed) sao mais")
print("  valiosos no mercado negro.\n")

# ---- G4: Histograma de total_loss_usd ----
print("G4: Distribuicao do prejuizo total...")
fig, ax = plt.subplots()
ax.hist(df_pd["total_loss_usd"].dropna(), bins=25, color="#1e6091", edgecolor="white", alpha=0.8)
mediana = df_pd["total_loss_usd"].median()
ax.axvline(mediana, color="#e63946", linewidth=2.5, linestyle="--", label=f"Mediana: {mediana:,.0f}")
ax.set_xlabel("total_loss_usd (USD)")
ax.set_ylabel("Frequencia")
ax.set_title("Distribuicao do Prejuizo Total (total_loss_usd)", fontweight="bold", fontsize=14)
ax.text(0.5, -0.12, "G4: Assimetria a direita — poucos incidentes com perdas muito altas",
        transform=ax.transAxes, ha="center", fontsize=10, color="gray", style="italic")
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig("graficos/grafico4_histograma_perdas.png", bbox_inches="tight")
plt.close()
print("  Grafico salvo: graficos/grafico4_histograma_perdas.png")
print("  INTERPRETACAO: A distribuicao e assimetrica a direita (cauda longa).")
print("  DECISAO: Usaremos a MEDIANA como referencia para o target binario.\n")

# ---- G5: Outliers no Prejuizo Total (IQR) ----
print("G5: Analise de outliers no prejuizo total...")
perdas = df_pd["total_loss_usd"].dropna().values
q1, q3 = np.percentile(perdas, [25, 75])
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr
outliers = perdas[perdas > limite_superior]
normais = perdas[perdas <= limite_superior]

fig, ax = plt.subplots()
ax.scatter(range(len(normais)), normais, c="#457b9d", s=15, alpha=0.6, label=f"Dados normais ({len(normais)})")
ax.scatter(range(len(normais), len(normais) + len(outliers)), outliers,
           c="#e63946", s=30, alpha=0.9, edgecolors="white", linewidth=0.5,
           label=f"Outliers IQR ({len(outliers)})")
ax.axhline(limite_superior, color="#e63946", linewidth=2, linestyle="--",
           label=f"Limite IQR: {limite_superior:,.0f}")
ax.set_xlabel("Indice do incidente")
ax.set_ylabel("Prejuizo (USD)")
ax.set_title("Outliers no Prejuizo Total (Metodo IQR)", fontweight="bold", fontsize=14)
ax.text(0.5, -0.12, "G5: Pontos em vermelho sao outliers acima do limite 1.5x IQR",
        transform=ax.transAxes, ha="center", fontsize=10, color="gray", style="italic")
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig("graficos/grafico5_outliers.png", bbox_inches="tight")
plt.close()
print("  Grafico salvo: graficos/grafico5_outliers.png")
print("  INTERPRETACAO: Pontos acima da linha vermelha sao OUTLIERS.")
print("  DECISAO: Na Gold, faremos clipping (capping) no limite superior do IQR.\n")

# ---- G6: Matriz de Correlacao ----
print("G6: Matriz de correlacao entre variaveis numericas...")
cols_num = ["total_loss_usd", "company_revenue_usd", "employee_count", "quality_score"]
corr_matrix = df_pd[cols_num].corr(method="pearson")

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
cmap = sns.diverging_palette(250, 15, s=75, l=40, n=12, center="light")
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap=cmap,
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.8,
            cbar_kws={"shrink": 0.75, "label": "Pearson r"})
ax.set_title("Matriz de Correlacao — Variaveis Numericas", fontweight="bold", fontsize=14)
ax.text(0.5, -0.08, "Pearson correlation coefficient",
        transform=ax.transAxes, ha="center", fontsize=10, color="gray", style="italic")
plt.tight_layout()
plt.savefig("graficos/grafico6_matriz_correlacao.png", bbox_inches="tight")
plt.close()
print("  Grafico salvo: graficos/grafico6_matriz_correlacao.png")
print("  INTERPRETACAO: Se total_loss_usd tem correlacao alta (>0.5)")
print("  com company_revenue_usd, empresas maiores sofrem perdas maiores.\n")

eda_time = time.time() - t0
tempos["EDA"] = eda_time
print(f"  [EDA] Concluida em {eda_time:.4f}s\n")


  EDA: ANALISE EXPLORATORIA DE DADOS


  Dataset: 778 linhas x 21 colunas

H1: Industrias de tecnologia sofrem mais ataques que outros setores?


  Grafico salvo: graficos/grafico1_top_industrias.png
  INTERPRETACAO: Se a industria 51 (technology) ou 52 (finance)
  liderar, confirma-se que setores com muitos dados digitais
  sao alvos prioritarios.

H2: Ransomware causa o maior prejuizo financeiro medio?


  Grafico salvo: graficos/grafico2_prejuizo_vetor.png
  INTERPRETACAO: Se ransomware lidera, justifica-se investir
  em protecao especifica (backups off-site, treinamento anti-phishing).

H3: Dados mistos (mixed) sao os mais visados por atacantes?


  Grafico salvo: graficos/grafico3_tipos_dados.png
  INTERPRETACAO: Dados financeiros + pessoais (mixed) sao mais
  valiosos no mercado negro.

G4: Distribuicao do prejuizo total...


  Grafico salvo: graficos/grafico4_histograma_perdas.png
  INTERPRETACAO: A distribuicao e assimetrica a direita (cauda longa).
  DECISAO: Usaremos a MEDIANA como referencia para o target binario.

G5: Analise de outliers no prejuizo total...


  Grafico salvo: graficos/grafico5_outliers.png
  INTERPRETACAO: Pontos acima da linha vermelha sao OUTLIERS.
  DECISAO: Na Gold, faremos clipping (capping) no limite superior do IQR.

G6: Matriz de correlacao entre variaveis numericas...


  Grafico salvo: graficos/grafico6_matriz_correlacao.png
  INTERPRETACAO: Se total_loss_usd tem correlacao alta (>0.5)
  com company_revenue_usd, empresas maiores sofrem perdas maiores.

  [EDA] Concluida em 2.2340s



## 🥇 Etapa 4: Camada Ouro (Gold) — ML-Ready com Fit/Transform
**Objetivo:** Preparar os dados para Machine Learning com o padrão FIT/TRANSFORM.

**FIT** = aprender parâmetros exclusivamente com dados de **TREINO** (80%)
**TRANSFORM** = aplicar esses parâmetros tanto no TREINO quanto no TESTE (20%)

Isso garante que **nenhuma informação do conjunto de teste vaze** para o treinamento.

**Transformações aplicadas:**
1. **Label Encoding** → confidence_tier (ordinal: 1<2<3<4)
2. **One-Hot Encoding** → industry_primary, attack_vector_primary, data_type (nominais)
3. **Mediana** → preenchimento de nulos numéricos (robusta a outliers)
4. **StandardScaler (Z-score)** → normalização de revenue e employee_count
5. **IQR Clipping** → total_loss_usd (cap 1.5×IQR)
6. **Z-score Clipping** → company_revenue_usd (cap \|z\|<3)


In [6]:
# ── OURO (GOLD) — FIT / TRANSFORM ──
print("=" * 60)
print("  OURO: PREPARACAO ML-READY (FIT/TRANSFORM)")
print("=" * 60)
t0 = time.time()

from pyspark.sql.functions import col
df_prata = spark.read.parquet("camada_prata/dataset_ml.parquet")
df_prata = df_prata.withColumn("total_loss_usd", col("total_loss_usd").cast("double"))
df_prata = df_prata.withColumn("company_revenue_usd", col("company_revenue_usd").cast("double"))
df_prata = df_prata.withColumn("employee_count", col("employee_count").cast("double"))
df_prata = df_prata.withColumn("quality_score", col("quality_score").cast("double"))
df_prata = df_prata.withColumn("confidence_tier", col("confidence_tier").cast("double"))

# Selecionar colunas uteis para o modelo (excluindo direct_loss_usd — data leakage)
colunas_manter = [
    "company_revenue_usd", "employee_count", "is_public_company",
    "industry_primary", "attack_vector_primary", "data_type",
    "confidence_tier", "quality_score", "total_loss_usd",
]
cols_existentes = [c for c in colunas_manter if c in df_prata.columns]
df_ml = df_prata.select(*cols_existentes)

df_pd_ml = df_ml.toPandas()
total = len(df_pd_ml)
split_idx = int(total * 0.8)

df_treino = df_pd_ml.iloc[:split_idx].copy()
df_teste  = df_pd_ml.iloc[split_idx:].copy()

print(f"  Split treino/teste: {len(df_treino)} treino, {len(df_teste)} teste (80/20)\n")

# ======================
#  FIT (apenas TREINO)
# ======================
print("  FIT: Aprendendo parametros no CONJUNTO DE TREINO...")

# 1. Categorias para One-Hot
onehot_specs = {}
for col_nome in ["industry_primary", "attack_vector_primary", "data_type"]:
    cats = sorted(df_treino[col_nome].dropna().unique())
    onehot_specs[col_nome] = cats

# 2. Mediana para colunas numericas
colunas_numericas = ["company_revenue_usd", "employee_count", "quality_score"]
median_values = {}
for c in colunas_numericas:
    median_values[c] = df_treino[c].median()

# 3. StandardScaler (Z-score): media e desvio
scaler_mean = {}
scaler_std = {}
for c in ["company_revenue_usd", "employee_count"]:
    scaler_mean[c] = df_treino[c].mean()
    scaler_std[c]  = df_treino[c].std()

# 4. IQR para outliers em total_loss_usd
q1 = df_treino["total_loss_usd"].quantile(0.25)
q3 = df_treino["total_loss_usd"].quantile(0.75)
iqr_upper = q3 + 1.5 * (q3 - q1)

# 5. Z-score para outliers em company_revenue_usd
rev_mean = df_treino["company_revenue_usd"].mean()
rev_std  = df_treino["company_revenue_usd"].std()
z_lower  = rev_mean - 3.0 * rev_std
z_upper  = rev_mean + 3.0 * rev_std

print(f"  FIT concluido! OneHot: {sum(len(v) for v in onehot_specs.values())} categorias, "
      f"Medianas: {len(median_values)}, Scaler: {len(scaler_mean)}, "
      f"IQR: {iqr_upper:.2f}, Z-score: [{z_lower:.2f}, {z_upper:.2f}]\n")

# ======================
#  TRANSFORM (aplica em ambos)
# ======================
def transform(df_part, params):
    df = df_part.copy()

    # 1. Label Encoding: confidence_tier
    df["confidence_tier"] = pd.to_numeric(df["confidence_tier"], errors="coerce").fillna(0).astype(int)

    # 2. One-Hot Encoding
    for col_nome, cats in params["onehot_specs"].items():
        for cat in cats:
            col_name = f"{col_nome}_{cat}"
            df[col_name] = (df[col_nome] == cat).astype(int)
    df.drop(columns=list(params["onehot_specs"].keys()), inplace=True, errors="ignore")

    # 3. Mediana para nulos numericos
    for c, med in params["median_values"].items():
        df[c].fillna(med, inplace=True)

    # 4. Standard Scaling
    for c in ["company_revenue_usd", "employee_count"]:
        mean = params["scaler_mean"][c]
        std  = params["scaler_std"][c]
        df[f"{c}_scaled"] = (df[c] - mean) / std if std > 0 else 0

    # 5. IQR Clipping em total_loss_usd
    iqr_up = params["iqr_upper"]
    df["total_loss_usd"] = df["total_loss_usd"].clip(upper=iqr_up)

    # 6. Z-score Clipping em company_revenue_usd
    df["company_revenue_usd"] = df["company_revenue_usd"].clip(lower=params["z_lower"], upper=params["z_upper"])

    return df

fit_params = {
    "onehot_specs": onehot_specs,
    "median_values": median_values,
    "scaler_mean": scaler_mean,
    "scaler_std": scaler_std,
    "iqr_upper": iqr_upper,
    "z_lower": z_lower,
    "z_upper": z_upper,
}

df_treino_t = transform(df_treino, fit_params)
df_teste_t  = transform(df_teste, fit_params)

df_ouro = pd.concat([df_treino_t, df_teste_t], axis=0).reset_index(drop=True)
df_ouro.to_parquet("camada_ouro/dataset_ml_ready.parquet", index=False)
print(f"  Dataset Ouro salvo: camada_ouro/dataset_ml_ready.parquet ({len(df_ouro)} linhas x {len(df_ouro.columns)} colunas)")

# Tabela de transformacoes
print("\n--- TABELA DE TRANSFORMACOES — CAMADA OURO ---")
print(f"  {'Transformacao':<22} | {'Tecnica':<25} | {'Colunas'}")
print(f"  {'-'*22}-+-{'-'*25}-+-{'-'*30}")
print(f"  {'Label Encoding':<22} | {'Cast str -> int32':<25} | confidence_tier")
print(f"  {'One-Hot Encoding':<22} | {'Dummy encoding':<25} | industry_primary ({len(onehot_specs.get('industry_primary',[]))} cats), attack_vector_primary, data_type")
print(f"  {'Missing (numerico)':<22} | {'Mediana (robusta)':<25} | company_revenue_usd, employee_count, quality_score")
print(f"  {'Missing (categorico)':<22} | {'Desconhecido':<25} | (feito na Prata)")
print(f"  {'Standard Scaling':<22} | {'Z-score (treino)':<25} | company_revenue_usd, employee_count")
print(f"  {'Outlier (IQR)':<22} | {'Clipping 1.5x IQR':<25} | total_loss_usd")
print(f"  {'Outlier (Z-score)':<22} | {'Clipping |z|<3':<25} | company_revenue_usd")
print("  Observacao: FIT usado dados de TREINO, TRANSFORM aplicado em ambos.")
print("  Nenhuma informacao do teste influenciou o aprendizado dos parametros.\n")

ouro_time = time.time() - t0
tempos["Ouro"] = ouro_time
print(f"  [OURO] Concluida em {ouro_time:.4f}s\n")


  OURO: PREPARACAO ML-READY (FIT/TRANSFORM)


  Split treino/teste: 622 treino, 156 teste (80/20)

  FIT: Aprendendo parametros no CONJUNTO DE TREINO...
  FIT concluido! OneHot: 36 categorias, Medianas: 3, Scaler: 2, IQR: 125420166.54, Z-score: [-54791975653.51, 76217668129.76]

  Dataset Ouro salvo: camada_ouro/dataset_ml_ready.parquet (778 linhas x 44 colunas)

--- TABELA DE TRANSFORMACOES — CAMADA OURO ---
  Transformacao          | Tecnica                   | Colunas
  -----------------------+---------------------------+-------------------------------
  Label Encoding         | Cast str -> int32         | confidence_tier
  One-Hot Encoding       | Dummy encoding            | industry_primary (20 cats), attack_vector_primary, data_type
  Missing (numerico)     | Mediana (robusta)         | company_revenue_usd, employee_count, quality_score
  Missing (categorico)   | Desconhecido              | (feito na Prata)
  Standard Scaling       | Z-score (treino)          | company_revenue_usd, employee_count
  Outlier (IQR)          | C

## 🤖 Etapa 5: Modelagem — Árvores de Decisão
**Objetivo:** Treinar 2 modelos de Árvore de Decisão para classificar incidentes como **ALTO IMPACTO** (prejuízo acima da mediana) ou **BAIXO IMPACTO**.

### Modelos:
- **Modelo 1:** Gini Index, `max_depth=5` (mais simples, evita overfitting)
- **Modelo 2:** Entropy, `max_depth=10` (mais complexo, divisões refinadas)

### Métricas:
- **Acurácia** = (VP + VN) / Total — % total de acertos
- **Precisão** = VP / (VP + FP) — quando diz "alto", quantas vezes acerta?
- **Recall** = VP / (VP + FN) — de todos os "alto" reais, quantos pegou?
- **F1-Score** = 2·P·R / (P+R) — média harmônica (balanceamento)

A **Matriz de Confusão** mostra VP, VN, FP, FN e permite calcular tudo.


### Comparação: Prata vs Ouro
Testamos os modelos com dados da **Camada Prata** (cru) e da **Camada Ouro** (tratado).
A diferença no F1-score mostra o **impacto do pré-processamento**!


In [7]:
# ── ML: TREINAMENTO E AVALIACAO ──
print("=" * 60)
print("  ML: MODELAGEM — ARVORES DE DECISAO")
print("=" * 60)
t0 = time.time()

# Carregar Prata (cru) e Ouro (tratado)
from pyspark.sql.functions import col
df_prata_ml = spark.read.parquet("camada_prata/dataset_ml.parquet")
df_prata_ml = df_prata_ml.withColumn("total_loss_usd", col("total_loss_usd").cast("double"))
df_prata_ml = df_prata_ml.withColumn("company_revenue_usd", col("company_revenue_usd").cast("double"))
df_prata_ml = df_prata_ml.withColumn("employee_count", col("employee_count").cast("double"))
df_prata_ml = df_prata_ml.withColumn("quality_score", col("quality_score").cast("double"))
df_prata_ml = df_prata_ml.withColumn("confidence_tier", col("confidence_tier").cast("double"))
df_ouro_ml  = pd.read_parquet("camada_ouro/dataset_ml_ready.parquet")

df_pd_prata = df_prata_ml.toPandas()
print(f"  Prata: {len(df_pd_prata)} linhas | Ouro: {len(df_ouro_ml)} linhas")

# ======================
# PREPARACAO DOS DADOS (PRATA)
# ======================
# Detectar features numericas dinamicamente (exceto total_loss_usd)
cols_numericas_orig = df_pd_prata.select_dtypes(include=[np.number]).columns.tolist()
feat_names_prata = [c for c in cols_numericas_orig if c not in ("total_loss_usd",)]
print(f"\n  Features detectadas (Prata): {len(feat_names_prata)} ({', '.join(feat_names_prata)})")

def preparar_dados(df, features):
    dados = df[features].fillna(0).values.astype(np.float64)
    loss_vals = df["total_loss_usd"].fillna(0).values.astype(np.float64)
    mediana = np.median(loss_vals)
    y = (loss_vals > mediana).astype(int)
    return dados, y, loss_vals, mediana

X, y, loss_vals, threshold = preparar_dados(df_pd_prata, feat_names_prata)
total = len(y)
split = int(total * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
loss_test = loss_vals[split:]

print(f"\n  CLASSIFICACAO: total_loss_usd > {threshold:.2f} = Alto Impacto (1) | <= {threshold:.2f} = Baixo Impacto (0)")
print(f"  Divisao Treino/Teste: 80/20 ({split} treino, {total - split} teste)\n")

# ======================
# MODELO 1: Gini Index, max_depth=5
# ======================
print("=" * 45)
print("  MODELO 1: Gini Index | max_depth=5")
print("  Modelo mais SIMPLES para evitar overfitting")
print("=" * 45)

m1 = DecisionTreeClassifier(criterion="gini", max_depth=5, random_state=42)
m1.fit(X_train, y_train)

pred1_train = m1.predict(X_train)
pred1_test  = m1.predict(X_test)

acc1_train = accuracy_score(y_train, pred1_train)
prec1_test = precision_score(y_test, pred1_test, zero_division=0)
rec1_test  = recall_score(y_test, pred1_test, zero_division=0)
f1_1       = f1_score(y_test, pred1_test, zero_division=0)
acc1_test  = accuracy_score(y_test, pred1_test)
f1_1_train = f1_score(y_train, pred1_train, zero_division=0)

print(f"  TREINO -> Acuracia: {acc1_train:.4f} | F1: {f1_1_train:.4f}")
print(f"  TESTE  -> Acuracia: {acc1_test:.4f} | F1: {f1_1:.4f}")

cm1 = confusion_matrix(y_test, pred1_test)
vp, fn_, fp, vn = cm1[1,1], cm1[1,0], cm1[0,1], cm1[0,0]
print(f"  MATRIZ: VP:{vp} FN:{fn_} FP:{fp} VN:{vn}")

# Feature Importance (Gini importance do sklearn)
importances_m1 = pd.DataFrame({
    "feature": feat_names_prata,
    "importance": m1.feature_importances_
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
cores = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(importances_m1)))
ax.barh(range(len(importances_m1)), importances_m1["importance"].values, color=cores)
ax.set_yticks(range(len(importances_m1)))
ax.set_yticklabels(importances_m1["feature"].values)
ax.set_xlabel("Importancia (Gini)")
ax.set_title("Importancia das Features — Modelo 1 (Gini, d=5)", fontweight="bold", fontsize=14)
sns.despine()
plt.tight_layout()
plt.savefig("graficos/feature_importance.png", bbox_inches="tight")
plt.close()
print("  Feature importance salva: graficos/feature_importance.png")

importances_m1.to_csv("previsoes/feature_importance.csv", index=False)
print("  Feature importance salva: previsoes/feature_importance.csv")

# ======================
# MODELO 2: Entropy, max_depth=10
# ======================
print("\n" + "=" * 45)
print("  MODELO 2: Entropy | max_depth=10")
print("  Modelo mais COMPLEXO, divisoes refinadas")
print("=" * 45)

m2 = DecisionTreeClassifier(criterion="entropy", max_depth=10, random_state=42)
m2.fit(X_train, y_train)

pred2_train = m2.predict(X_train)
pred2_test  = m2.predict(X_test)

acc2_train = accuracy_score(y_train, pred2_train)
prec2_test = precision_score(y_test, pred2_test, zero_division=0)
rec2_test  = recall_score(y_test, pred2_test, zero_division=0)
f1_2       = f1_score(y_test, pred2_test, zero_division=0)
acc2_test  = accuracy_score(y_test, pred2_test)
f1_2_train = f1_score(y_train, pred2_train, zero_division=0)

print(f"  TREINO -> Acuracia: {acc2_train:.4f} | F1: {f1_2_train:.4f}")
print(f"  TESTE  -> Acuracia: {acc2_test:.4f} | F1: {f1_2:.4f}")

cm2 = confusion_matrix(y_test, pred2_test)
vp2, fn2_, fp2, vn2 = cm2[1,1], cm2[1,0], cm2[0,1], cm2[0,0]
print(f"  MATRIZ: VP:{vp2} FN:{fn2_} FP:{fp2} VN:{vn2}")

# ======================
# MATRIZ DE CONFUSAO DO MELHOR MODELO
# ======================
melhor = 1 if f1_2 >= f1_1 else 2
cm_best = cm2 if melhor == 1 else cm1
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_best, annot=True, fmt="d", cmap="RdYlGn", cbar=False,
            xticklabels=["Baixo (0)", "Alto (1)"],
            yticklabels=["Baixo (0)", "Alto (1)"])
ax.set_xlabel("Previsto")
ax.set_ylabel("Real")
ax.set_title(f"Matriz de Confusao — Modelo {melhor} (d={5 if melhor==1 else 10})", fontweight="bold")
plt.tight_layout()
plt.savefig("graficos/matriz_confusao.png", bbox_inches="tight")
plt.close()
print(f"\n  Matriz de confusao salva: graficos/matriz_confusao.png (Modelo {melhor})")

# ======================
# SALVAR PREVISOES
# ======================
def salvar_previsoes(y_t, pred, losses, thresh, prefixo):
    df_out = pd.DataFrame({
        "id": range(len(y_t)),
        "real": ["Alto" if r == 1 else "Baixo" for r in y_t],
        "previsto": ["Alto" if p == 1 else "Baixo" for p in pred],
        "acertou?": ["SIM" if r == p else "NAO" for r, p in zip(y_t, pred)],
        "total_loss_usd": losses,
    })
    path = f"previsoes/previsoes_{prefixo}.csv"
    df_out.to_csv(path, index=False)
    print(f"  Previsoes salvas: {path}")

salvar_previsoes(y_test, pred1_test, loss_test, threshold, "modelo1")
salvar_previsoes(y_test, pred2_test, loss_test, threshold, "modelo2")

# CSV comparativo
comp = pd.DataFrame({
    "id": range(len(y_test)),
    "real": ["Alto" if r == 1 else "Baixo" for r in y_test],
    "modelo1_gini": ["Alto" if p == 1 else "Baixo" for p in pred1_test],
    "modelo2_entropy": ["Alto" if p == 1 else "Baixo" for p in pred2_test],
    "acertou_1?": ["SIM" if y_test[i]==pred1_test[i] else "NAO" for i in range(len(y_test))],
    "acertou_2?": ["SIM" if y_test[i]==pred2_test[i] else "NAO" for i in range(len(y_test))],
    "total_loss_usd": loss_test,
})
comp.to_csv("previsoes/comparacao_modelos.csv", index=False)
print("  Comparacao salva: previsoes/comparacao_modelos.csv")

# Salvar modelo como JSON
modelo_json = {
    "modelo": "DecisionTreeClassifier",
    "criterio": ["Gini", "Entropy"],
    "max_depth": [5, 10],
    "features": feat_names_prata,
    "metricas_modelo1": {
        "acuracia_treino": round(acc1_train, 4),
        "f1_treino": round(f1_1_train, 4),
        "acuracia_teste": round(acc1_test, 4),
        "f1_teste": round(f1_1, 4),
        "n_treino": split,
        "n_teste": total - split,
    },
    "metricas_modelo2": {
        "acuracia_treino": round(acc2_train, 4),
        "f1_treino": round(f1_2_train, 4),
        "acuracia_teste": round(acc2_test, 4),
        "f1_teste": round(f1_2, 4),
        "n_treino": split,
        "n_teste": total - split,
    }
}
with open("modelos/modelo_gini_entropy.json", "w") as f:
    json.dump(modelo_json, f, indent=2)
print("  Modelos salvos: modelos/modelo_gini_entropy.json")

# ======================
# COMPARACAO: PRATA vs OURO
# ======================
print("\n" + "=" * 45)
print("  COMPARACAO: PRATA vs OURO")
print("=" * 45)

# Detectar features na Ouro (todas as colunas numericas exceto total_loss_usd)
feat_ouro = [c for c in df_ouro_ml.select_dtypes(include=[np.number]).columns.tolist()
             if c not in ("total_loss_usd",)]
print(f"\n  Prata: {len(feat_names_prata)} features | Ouro: {len(feat_ouro)} features (com one-hot)")

X_ouro, y_ouro, _, _ = preparar_dados(df_ouro_ml, feat_ouro)
X_train_ouro, X_test_ouro = X_ouro[:split], X_ouro[split:]
y_train_ouro, y_test_ouro = y_ouro[:split], y_ouro[split:]

m_ouro = DecisionTreeClassifier(criterion="entropy", max_depth=10, random_state=42)
m_ouro.fit(X_train_ouro, y_train_ouro)
pred_ouro = m_ouro.predict(X_test_ouro)

acc_ouro = accuracy_score(y_test_ouro, pred_ouro)
f1_ouro  = f1_score(y_test_ouro, pred_ouro, zero_division=0)

print("\n  COMPARATIVO FINAL:")
print("  " + "-" * 55)
print(f"  {'Metrica':<15} | {'Prata (cru)':<15} | {'Ouro (tratado)':<15} | {'Ganhou?'}")
print("  " + "-" * 55)
df = f1_ouro - f1_2
da = acc_ouro - acc2_test
print(f"  {'F1-Score':<15} | {f1_2:<15.4f} | {f1_ouro:<15.4f} | {'SIM' if df > 0.001 else '=' if df >= 0 else 'NAO'}")
print(f"  {'Acuracia':<15} | {acc2_test:<15.4f} | {acc_ouro:<15.4f} | {'SIM' if da > 0.001 else '=' if da >= 0 else 'NAO'}")
print("  " + "-" * 55)

if df > 0.001:
    print("\n  CONCLUSAO: O pre-processamento da Ouro MELHOROU o modelo!")
elif df >= 0:
    print("\n  CONCLUSAO: O pre-processamento manteve o F1 igual, MAS")
    print("      o modelo e mais ROBUSTO e GENERALIZAVEL.")
else:
    print("\n  CONCLUSAO: O pre-processamento nao melhorou o F1, mas")
    print("      o modelo e mais ROBUSTO e GENERALIZAVEL.")

print("\n  ARVORE DE DECISAO — ESTRUTURA:")
print("   O Modelo 1 (Gini, d=5) cria ate 5 niveis de profundidade.")
print("   Cada no testa uma feature (ex: company_revenue_usd > X)")
print("   cada folha: Alto Impacto (1) ou Baixo Impacto (0).")
print("   NOTA: direct_loss_usd foi removida (era componente do target).\n")

ml_time = time.time() - t0
tempos["ML"] = ml_time
print(f"  [ML] Concluida em {ml_time:.4f}s\n")


  ML: MODELAGEM — ARVORES DE DECISAO


  Prata: 778 linhas | Ouro: 778 linhas

  Features detectadas (Prata): 4 (company_revenue_usd, employee_count, confidence_tier, quality_score)

  CLASSIFICACAO: total_loss_usd > 16564914.86 = Alto Impacto (1) | <= 16564914.86 = Baixo Impacto (0)
  Divisao Treino/Teste: 80/20 (622 treino, 156 teste)

  MODELO 1: Gini Index | max_depth=5
  Modelo mais SIMPLES para evitar overfitting
  TREINO -> Acuracia: 0.6849 | F1: 0.6512
  TESTE  -> Acuracia: 0.5577 | F1: 0.4889
  MATRIZ: VP:33 FN:45 FP:24 VN:54
  Feature importance salva: graficos/feature_importance.png
  Feature importance salva: previsoes/feature_importance.csv

  MODELO 2: Entropy | max_depth=10
  Modelo mais COMPLEXO, divisoes refinadas


  TREINO -> Acuracia: 0.7621 | F1: 0.7289
  TESTE  -> Acuracia: 0.5000 | F1: 0.4091
  MATRIZ: VP:27 FN:51 FP:27 VN:51

  Matriz de confusao salva: graficos/matriz_confusao.png (Modelo 2)
  Previsoes salvas: previsoes/previsoes_modelo1.csv
  Previsoes salvas: previsoes/previsoes_modelo2.csv
  Comparacao salva: previsoes/comparacao_modelos.csv
  Modelos salvos: modelos/modelo_gini_entropy.json

  COMPARACAO: PRATA vs OURO

  Prata: 4 features | Ouro: 42 features (com one-hot)

  COMPARATIVO FINAL:
  -------------------------------------------------------
  Metrica         | Prata (cru)     | Ouro (tratado)  | Ganhou?
  -------------------------------------------------------
  F1-Score        | 0.4091          | 0.5641          | SIM
  Acuracia        | 0.5000          | 0.5641          | SIM
  -------------------------------------------------------

  CONCLUSAO: O pre-processamento da Ouro MELHOROU o modelo!

  ARVORE DE DECISAO — ESTRUTURA:
   O Modelo 1 (Gini, d=5) cria ate 5 niveis de

## ⏱️ Relatório de Tempos
Tempos de execução de cada etapa do pipeline PySpark, salvos em `comparacao_tempos/tempos_pyspark.csv`.


In [8]:
# ── SALVAR TEMPOS ──
print("=" * 60)
print("  RELATORIO DE TEMPOS")
print("=" * 60)

tempos_df = pd.DataFrame([
    {"projeto": "PySpark", "etapa": etapa, "tempo_segundos": round(t, 4)}
    for etapa, t in tempos.items()
])
total = tempos_df["tempo_segundos"].sum()
tempos_df = pd.concat([
    tempos_df,
    pd.DataFrame([{"projeto": "PySpark", "etapa": "total", "tempo_segundos": round(total, 4)}])
], ignore_index=True)

tempos_df.to_csv(os.path.join(COMPARACAO, "tempos_pyspark.csv"), index=False)
print(f"\n  Tempos salvos em: {os.path.join(COMPARACAO, 'tempos_pyspark.csv')}")

print("\n  " + "-" * 45)
print(f"  {'Etapa':<15} | {'Tempo (s)':<15}")
print("  " + "-" * 45)
for _, row in tempos_df.iterrows():
    print(f"  {row['etapa']:<15} | {row['tempo_segundos']:<15.4f}")
print("  " + "-" * 45)

print(f"\n  ⏱  Tempo TOTAL: {total:.4f}s")
print("  [PIPELINE PySpark COMPLETO COM SUCESSO!]\n")

spark.stop()


  RELATORIO DE TEMPOS

  Tempos salvos em: /home/ben/Área de trabalho/Projeto Ciencia de dados/projeto_pyspark/../comparacao_tempos/tempos_pyspark.csv

  ---------------------------------------------
  Etapa           | Tempo (s)      
  ---------------------------------------------
  Bronze          | 9.3323         
  Prata           | 7.1001         
  EDA             | 2.2340         
  Ouro            | 0.4943         
  ML              | 0.7571         
  total           | 19.9178        
  ---------------------------------------------

  ⏱  Tempo TOTAL: 19.9178s
  [PIPELINE PySpark COMPLETO COM SUCESSO!]



---
**Pipeline Medallion** — Bronze → Prata → Ouro → ML
**Aluno:** Benjamin Yuji Suzuki
**Disciplina:** Ciência de Dados
**Implementação:** PySpark com sklearn DecisionTreeClassifier
